
# Week 1 — Introduction to AI Engineering

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/lectures/intro_lecture.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

📘 **Theme:** From Algorithms → Systems → Reliability  


---

### **Learning Objectives**
By the end of this week, you will be able to:
1. Explain what *AI Engineering* means and how it differs from algorithmic AI.
2. Describe real-world successes and limitations of large language models (LLMs).
3. Interpret key failure types: hallucinations, bias, brittleness.
4. Build a simple mental model of how LLMs generate text.
5. Understand the *Unifying System Diagram* of LLM systems.
6. Run your first API call and reason about it scientifically.
7. Reflect on reliability, trust, and the iterative design mindset.


In [1]:
# @title Setup (Run this first)
!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git
import sys; sys.path.append("/content/main")
from course_utils import lab1_setup, show_mermaid

lab1_setup()
print("✅ Environment ready!")

🔧 Setting up your environment...
  → Installing core packages...
installing mermaid-python
  → Setting random seed for reproducible results...
  → Checking API key...
🔑 Enter your OpenAI API key.
   (It will only be stored in this Colab runtime - it's safe!)
   Get your key from: https://platform.openai.com/api-keys
OpenAI API key: ··········
✅ API key set.
  → Adding course files to path...
✅ Setup complete!
✅ lab1_setup: environment ready.
✅ Environment ready!



## 🧩 **Day 1 — What Does It Mean to Engineer AI?**
---
**Guiding question:**  
> How is AI Engineering different from building models, and why does that matter?




### Welcome & Motivation


> “Who here has used ChatGPT or another AI tool this week?”

<br><br><br>

> “When it worked well, why? When did it fail?”

<br><br><br>

> “That gap — between impressive and unreliable — is what AI engineers work to close.”

**AI Engineering = Designing systems that are repeatable, safe, and measurable.**



### What Is AI Engineering (and How Is It Different)?

| Course | Focus | Core Question |
|:--------|:--------|:-------------|
| *Intro to AI* | Symbolic reasoning, search | “How do we find the best move?” |
| *Intro to Deep Learning* | Model architectures | “How does a CNN learn features?” |
| *NLP* | Linguistic representation | “How can we classify text?” |
| **AI Engineering** | System reliability, safety | “How can we make AI systems reliable and auditable?” |

> AI Engineering bridges models, systems, and people. Engineers design pipelines, test behaviors, and trade off between accuracy, latency, and safety.



### LLM Successes and Failures

We'll contrast **impressive** and **unreliable** examples to motivate why *engineering* matters.


In [6]:
# @title Example 1 — Helpful Assistant
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain how a neural network recognizes handwriting, in one paragraph."}
    ]
)
display(response.choices[0].message.content)

"A neural network recognizes handwriting by processing image data through multiple layers of interconnected neurons, utilizing techniques like feature extraction and pattern recognition. Initially, the input image of handwritten text is transformed into a numerical format, often through pixel intensities. The network's layers, particularly convolutional layers, identify key features such as curves and edges by applying filters that highlight different patterns. As the data passes through successive layers, the network learns to recognize more complex representations, integrating information about shapes and structures. Ultimately, the final layers produce probabilities for different character classifications, allowing the network to identify letters and words effectively. Through training on extensive datasets of labeled handwriting, the neural network continuously adjusts its parameters to improve accuracy and adaptability in recognizing varied handwriting styles."

In [5]:
# @title Example 2 — Confidently Wrong Model (Hallucination)
response = client.chat.completions.create(
    model="gpt-3.5-turbo-0125",
    messages=[{"role": "user", "content": "Explain in 50 words or less how the Great Wall of China blocks satellite signals"}]
)
display(response.choices[0].message.content)
display("💭 Note: This is a hallucination — the model confidently states something that is false.")
display("   The Great Wall does NOT block satellite signals. This is why we need to test and verify AI outputs!")

'The Great Wall of China is made of dense materials like stone and bricks that can interfere with satellite signals, causing disruptions or even completely blocking transmissions. The thick and tall structure creates a physical barrier that can obstruct the line of sight between the satellite and its intended destination.'

'\n💭 Note: This is a hallucination — the model confidently states something that is false.'

'   The Great Wall does NOT block satellite signals. This is why we need to test and verify AI outputs!'


### 🔤 Building a Mental Model of LLMs

Think of an LLM as an *autocomplete engine on steroids* — predicting what comes next, token by token.

**Key insight:** The model doesn't "know" facts — it predicts what text is likely to come next based on patterns it learned from training data. This is why it can be both impressive and unreliable.


In [8]:
from IPython.display import Markdown
prompt = "Artificial intelligence is..."
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)
display(Markdown(response.choices[0].message.content))

Artificial intelligence (AI) is a branch of computer science focused on creating systems that can perform tasks typically requiring human intelligence. These tasks include reasoning, problem-solving, learning, understanding natural language, recognizing patterns, and making decisions. AI can be divided into two main categories:

1. **Narrow AI**: Systems designed and trained for specific tasks, such as language translation, image recognition, or playing chess. This is the type of AI most commonly encountered today.

2. **General AI**: A theoretical form of AI that possesses the ability to understand, learn, and apply intelligence across a wide range of tasks, similar to a human. This type of AI is still largely hypothetical and has not been achieved.

AI technologies encompass various subfields, including machine learning (where algorithms improve through experience), natural language processing (enabling machines to understand and generate human language), computer vision (enabling machines to interpret visual information), and robotics. AI has applications across numerous sectors, including healthcare, finance, transportation, entertainment, and more, transforming how tasks are performed and revolutionizing industries. 

As AI continues to evolve, it raises important ethical, social, and economic considerations regarding its impact on society, privacy, and the future of work.


### 🧠 The Unifying System Diagram

This 5-part diagram will guide us through the course:

**User Interaction → Prompt & Control → Tools & Augmentation → Core LLM → Output & Monitoring**



<!-- ![LLM System Diagram](https://github.com/tulane-intro-ai-engineering/main/blob/main/lectures/llm_workflow.png?raw=true) -->




In [ ]:
# @title Detailed System Diagram

show_mermaid("""
graph TD
    subgraph User Interaction
    U["👤 Users<br/>Queries / Inputs"]:::user --> IH["Input Handling<br/>• Formatting<br/>• Validation<br/>• Safety Filters"]:::process
    end

    subgraph Prompt & Control
    IH --> PC("Prompt / Control<br/>• Instructions<br/>• Examples<br/>• Constraints<br/>• Parameters"):::control
    end

    subgraph Tools & Augmentation
    PC --> TF{"Tools / Functions<br/>• External APIs"}:::tool
    PC --> RAG{"Retrieval (RAG)<br/>• Embeddings<br/>• Vector Store<br/>• Top-k Search"}:::tool
    end

    subgraph Core LLM
    TF --> LLM["Core LLM<br/>• Next-token generation<br/>• Sampling<br/>• Fine-tuned model"]:::model
    RAG --> LLM
    PC --> LLM
    end

    subgraph Output & Monitoring
    LLM --> OP["Output Processing<br/>• Formatting<br/>• Citations<br/>• Refusals<br/>• Trust Signals"]:::output --> O("Final Output"):::output
    O --> LM["Logging & Monitoring<br/>• Prompts & Responses<br/>• Metrics<br/>• Drift Detection"]:::monitor
    end

    classDef user fill:#d1e7dd,stroke:#333,stroke-width:1px;
    classDef process fill:#e2e3e5,stroke:#333,stroke-width:1px;
    classDef control fill:#cfe2ff,stroke:#333,stroke-width:1px;
    classDef tool fill:#fff3cd,stroke:#333,stroke-width:1px;
    classDef model fill:#f8d7da,stroke:#333,stroke-width:1px;
    classDef output fill:#e9ecef,stroke:#333,stroke-width:1px;
    classDef monitor fill:#fefefe,stroke:#333,stroke-width:1px;
""")

**Where do you think reliability issues arise most often?**

<br><br><br>



### 🤝 Trust Activity

Scenario brainstorming (small groups):  
- Would you trust AI for medical advice? grading essays? writing policy?  
Mark which *stages* of the pipeline you’d trust vs. audit.

**Discussion:** What common patterns emerge?



### Course Overview & Lab 1 Preview

- Weekly rhythm: Tues = concept, Thurs = lab.  
- Labs = *mini scientific investigations.*  
- **Lab 1:** make your first API call, measure model behavior.

> “Next time, we’ll talk directly to this system — through an API.”



## 💻 **Day 2 — From Concept to Code: APIs and the Scientific Method**

**Guiding question:**  
> How do we interact with an AI system — and test it like scientists?



### 🌐 What Is an API?

Analogy: ordering from a restaurant menu — you don’t enter the kitchen, you make a request.

Diagram:
```
User → Request (JSON) → Server → Model → Response (JSON)
```


In [9]:
# Example API request
example_request = {
    "model": "gpt-4o-mini",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of Japan?"}
    ]
}
example_request

{'model': 'gpt-4o-mini',
 'messages': [{'role': 'system', 'content': 'You are a helpful assistant.'},
  {'role': 'user', 'content': 'What is the capital of Japan?'}]}


### ⚡ Live Demo: Hello API


In [10]:
response = client.chat.completions.create(**example_request)
print(response.choices[0].message.content)
print("Tokens used:", response.usage.total_tokens)

The capital of Japan is Tokyo.
Tokens used: 31


In [11]:
print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-CwCd5tiBFFILZ81x3LU7SmUciqwgh",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "The capital of Japan is Tokyo.",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1767987607,
  "model": "gpt-4o-mini-2024-07-18",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_29330a9688",
  "usage": {
    "completion_tokens": 7,
    "prompt_tokens": 24,
    "total_tokens": 31,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cached_tokens": 0
    }
  }
}



### 🧪 Mini Experiment: Prompt Wording and Response Length

We'll test whether prompt phrasing changes the output length.


In [12]:
prompts = [
    "Explain AI engineering in one sentence.",
    "Explain AI engineering in one sentence using technical language."
]

for p in prompts:
    resp = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": p}])
    text = resp.choices[0].message.content
    print(f"Prompt: {p}")
    display(Markdown(text))
    print(f"Length: {len(text)} characters\n")

Prompt: Explain AI engineering in one sentence.


AI engineering is the discipline that involves designing, building, and deploying AI systems and algorithms to solve real-world problems effectively and efficiently.

Length: 165 characters

Prompt: Explain AI engineering in one sentence using technical language.


AI engineering is the multidisciplinary field that encompasses the design, development, and deployment of complex algorithms and architectures utilizing machine learning, data processing, and systems integration to create intelligent applications and automated solutions.

Length: 271 characters




### 🧭 Responsible Iteration & Measurement

**AI engineers think in loops:**
1. Observe model behavior.
2. Adjust prompt or parameters.
3. Measure results.
4. Reflect and repeat.

This is not “prompt hacking” — it’s controlled experimentation.



### 🧫 Introducing Lab 1

**What you'll do in Lab 1:**
- Make your first API call to an LLM
- **Experiment** with different system prompts (using the scientific method!)
- Build a simple web app with Gradio
- Observe how system prompts affect model behavior

**Lab structure:**
- **Setup** (clone repo + bootstrap)
- **Pre-Lab** (conceptual warmup)
- **Scientific Process** (Question → Hypothesis → Experiment → Measurement)
- **Experiment** (test different system prompts)
- **Results & Reflection** (connect to reliability)

**Connection to today's lecture:** In the lab, you'll apply the scientific method we just discussed to test how system prompts change model outputs.



### 👩‍💻 In-Class Lab Work

Students launch Lab 1 in Colab, test API, record first measurements.

**Exit Ticket:**  
> “What surprised you about the model’s response today?”



<details>
<summary>🧑‍🏫 Instructor Notes</summary>

**Pacing:**  
- Demos should be quick; skip reruns if latency >15s.  
- Prioritize discussion over full code explanations.  
- If students struggle with setup, pause and debug as a group.

**Engagement Tips:**  
- Use polls for trust activities.  
- Encourage sharing examples of “good/bad” AI behavior.

**Extensions:**  
- Optional demo: temperature or top_p for creativity.  
- Ask students to predict which prompt will be longer *before* running it.
</details>
